In [0]:
%skip
%sql
ALTER EXTERNAL LOCATION `adls-location` RENAME TO `adls-landing-location`;

In [0]:
%skip
dbutils.fs.ls(f"/Volumes/adb_healthcare_app_7405606064149341/landing/landing_volume/{source_system}/{table_name}/{landing_timestamp}/")

In [0]:
# Create widgets
dbutils.widgets.text("source_system", "", "Source System")
dbutils.widgets.text("table_name", "", "Table Name")
dbutils.widgets.text("landing_timestamp", "", "Landing Timestamp")
dbutils.widgets.text("source_file_format", "", "Source File Format")

# Get widget values into variables
source_system = dbutils.widgets.get("source_system")
table_name = dbutils.widgets.get("table_name")
landing_timestamp = dbutils.widgets.get("landing_timestamp")
source_file_format = dbutils.widgets.get("source_file_format")

In [0]:
base_landing_path = "/Volumes/adb_healthcare/landing/landing_volume"
source_path = f"{base_landing_path}/{source_system}"
table_path = f"{source_path}/{table_name}"
landing_path = f"{table_path}/{landing_timestamp}/"

def path_exists(path: str) -> bool:
  try:
    dbutils.fs.ls(path)
    return True
  except Exception:
    return False

missing_path = next((path for path in [source_path, table_path, landing_path] if not path_exists(path)), None)

if missing_path:
  message = (
      f"No new files are received for source_system = {source_system} and table_name = {table_name} and landing_timestamp = {landing_timestamp}.\n"
      f"Please confirm that the source system is sending data to {missing_path}"
  )
  print(message)
  dbutils.notebook.exit(f"ERROR: {message}")


In [0]:
if source_file_format == "csv":
    landing_df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(landing_path)
else:
    landing_df = spark.read.format("parquet").load(landing_path)

In [0]:
from pyspark.sql import functions as F

updated_df = (
    landing_df
        .withColumn("landing_timestamp", F.lit(landing_timestamp))
        .withColumn("insert_timestamp", F.current_timestamp())
)

In [0]:
%skip
display(landing_df)
display(updated_df)

In [0]:
updated_df.write.mode("append").saveAsTable(f"adb_healthcare.bronze.{table_name}")

In [0]:
%sql
--truncate table adb_healthcare.bronze.hospitals
select * from adb_healthcare.bronze.hospitals
--show catalogs